# Persian G2P (Grapheme-to-Phoneme) with ByT5-small on KaamelDict

This notebook fine-tunes `google/byt5-small` to convert Persian text (orthography) into space-separated phonetic representations (IPA/phonemes) using the **KaamelDict** dataset (~116k entries).

### Key Technical Settings:
- **Full Precision (FP32)**: Avoids known `fp16` underflow / NaN loss issues on T5/ByT5 architectures.
- **Gradient Accumulation (steps=2)**: Maintains effective batch size of 32 while preserving VRAM headroom.
- **Fast Evaluation**: `predict_with_generate=False` during epoch evaluation to prevent slow autoregressive beam searches.
- **Google Drive Persistence**: Checkpoints and the final model are saved directly to your 5TB Google Drive.

In [ ]:
# Step 1: Install Required Libraries
!pip install -q transformers datasets accelerate evaluate pandas

In [ ]:
# Step 2: Check GPU and Mount Google Drive
import os
import torch
from google.colab import drive

assert torch.cuda.is_available(), "No GPU found! Go to Runtime > Change runtime type > Select T4 GPU."
print(f"Active GPU: {torch.cuda.get_device_name(0)}")

# Mount Google Drive
drive.mount('/content/drive')

# Destination path inside your 5TB Google Drive
GDRIVE_SAVE_DIR = "/content/drive/MyDrive/models/persian-byt5-g2p"
os.makedirs(GDRIVE_SAVE_DIR, exist_ok=True)
print(f"Models will be saved permanently to: {GDRIVE_SAVE_DIR}")

In [ ]:
# Step 3: Load and Preprocess KaamelDict Dataset
import ast
import pandas as pd
from datasets import Dataset

print("Loading KaamelDict CSV dataset from Hugging Face...")
CSV_URL = "https://huggingface.co/datasets/MahtaFetrat/KaamelDict/resolve/main/KaamelDict.csv"
df = pd.read_csv(CSV_URL)

def format_phonemes(phoneme_raw):
    if pd.isna(phoneme_raw):
        return ""
    try:
        data = ast.literal_eval(str(phoneme_raw))
        if isinstance(data, list) and len(data) > 0:
            # Take primary pronunciation (first tuple) and space-separate
            return " ".join(data[0])
    except Exception:
        pass
    return ""

df["target"] = df["phoneme"].apply(format_phonemes)
df = df.dropna(subset=["grapheme"])
df = df[df["target"].str.strip() != ""][["grapheme", "target"]].rename(columns={"grapheme": "input"})
df["input"] = df["input"].astype(str)

print(f"Total valid phonetic pairs: {len(df):,}")
print("Sample entries:")
display(df.head(5))

# Train/Validation Split (95% Train, 5% Test)
raw_dataset = Dataset.from_pandas(df).train_test_split(test_size=0.05, seed=42)
print(f"Train size: {len(raw_dataset['train']):,} | Eval size: {len(raw_dataset['test']):,}")

In [ ]:
# Step 4: Tokenization via ByT5 (Byte-Level)
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_ID = "google/byt5-small"
print(f"Loading tokenizer and model: {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)

def preprocess_function(examples):
    # Max 64 UTF-8 bytes for Persian input word
    model_inputs = tokenizer(examples["input"], max_length=64, truncation=True)
    # Max 128 UTF-8 bytes for space-separated phoneme output
    labels = tokenizer(examples["target"], max_length=128, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Tokenizing dataset...")
tokenized_datasets = raw_dataset.map(
    preprocess_function, 
    batched=True, 
    remove_columns=raw_dataset["train"].column_names
)
print("Tokenization complete!")

In [ ]:
# Step 5: Training Setup (FP32 + Gradient Accumulation + Fast Loss Eval)
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

training_args = Seq2SeqTrainingArguments(
    output_dir="./local_checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,     # Effective batch size = 32
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=1,
    fp16=False,                        # FP32: Prevents T5/ByT5 NaN loss / underflow on T4
    bf16=False,
    predict_with_generate=False,       # Fast loss-only eval during training
    logging_steps=100,
    warmup_steps=300,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
)

In [ ]:
# Step 6: Start Training & Save to Google Drive
print("Starting fine-tuning on T4 GPU...")
trainer.train()

print(f"\nTraining complete! Exporting model to Google Drive at: {GDRIVE_SAVE_DIR}")
trainer.save_model(GDRIVE_SAVE_DIR)
tokenizer.save_pretrained(GDRIVE_SAVE_DIR)
print("Model and tokenizer successfully saved in Google Drive!")

In [ ]:
# Step 7: Live Inference Verification
def predict_g2p(word: str) -> str:
    inputs = tokenizer(word, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=128,
            num_beams=2,
            early_stopping=True
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

test_words = [
    "دانشگاه",
    "کامپیوتر",
    "خوش‌آمدید",
    "هوش مصنوعی",
    "آسمان",
    "تلفظ",
    "واترپولو",
    "دربازکن"
]

print("\n--- G2P Inference Verification ---")
for w in test_words:
    print(f"{w:15} -> {predict_g2p(w)}")